# SoundStream Neural Codec Demo



SoundStream replica demo: load a trained neural audio codec, prepare an audio sample, reconstruct it, and inspect the encoded representation.


## Setup

The setup cell resolves the project root, configures local plotting cache, and makes repository modules importable.


In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone git@github.com:Ulorew/NeuralCodec.git
    %cd NeuralCodec/

In [ ]:
import os
from pathlib import Path
import sys


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").is_file() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("Could not find NeuralCodecDemo project root")


PROJECT_ROOT = find_project_root()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache" / "matplotlib"))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


> Runtime note: default dependencies target CPU PyTorch; `requirements_cuda.txt` contains the CUDA variant for RTX 50-series / `sm_120`.


In [ ]:
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import requests
import torch
import torchaudio
import torch.nn.functional as F
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from IPython.display import Audio, display

from src.transforms import Normalize1D
from src.utils.io_utils import ROOT_PATH


## 1. Select Inputs

The demo accepts local paths or HTTP(S) URLs for the checkpoint and audio sample. Clip controls can limit the evaluated audio segment.


In [ ]:
MODEL_SOURCE = "https://huggingface.co/buckets/ulorew/NeuralCodec/resolve/gan_training_bit6_200ep_0.5s_mixed_loss_C32_great_best.pth?download=true"
AUDIO_SOURCE = "https://huggingface.co/buckets/ulorew/NeuralCodec/resolve/helloeveryone%E2%80%8B.flac?download=true"

TARGET_SR = 16_000
CLIP_SECONDS = None
START_SECONDS = 0.0

DEMO_DATA_DIR = PROJECT_ROOT / "demo_data"
DEMO_DATA_DIR.mkdir(exist_ok=True)

In [ ]:
def is_url(source):
    return urlparse(str(source)).scheme in {"http", "https"}


def resolve_input(source, dst_dir=DEMO_DATA_DIR, filename=None):
    source = str(source)
    if is_url(source):
        name = filename or Path(urlparse(source).path).name
        dst = dst_dir / name
        if not dst.exists():
            with requests.get(source, stream=True, timeout=60) as response:
                response.raise_for_status()
                with dst.open("wb") as handle:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            handle.write(chunk)
        return dst

    src = Path(source).expanduser()
    if not src.exists():
        raise FileNotFoundError(
            f"Input not found: {src}. Expected a valid local path or URL."
        )
    return src


def select_device():
    if not torch.cuda.is_available():
        return torch.device("cpu")

    return torch.device("cuda")


def display_path(path):
    path = Path(path)
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def load_checkpoint(model, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state_dict = checkpoint.get("state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
    model.load_state_dict(state_dict)
    return checkpoint


## 2. Load Model

The model architecture comes from `src/configs/model/soundstream.yaml`; the checkpoint provides trained weights.


In [ ]:
model_path = resolve_input(MODEL_SOURCE, filename="model_best.pth")
device = select_device()

config_dir = str((ROOT_PATH / "src/configs").resolve())
with initialize_config_dir(version_base=None, config_dir=config_dir):
    cfg = compose(config_name="inference")

model = instantiate(cfg.model).to(device)
checkpoint = load_checkpoint(model, model_path, device)
model.eval()

epoch = checkpoint.get("epoch", "unknown") if isinstance(checkpoint, dict) else "raw state_dict"
print(f"Loaded checkpoint: {display_path(model_path)}  \nEpoch: {epoch}  \nDevice: {device}")


## 3. Prepare Audio

The codec expects mono 16 kHz audio. The preparation step converts channels, resamples, selects the requested segment, aligns the length to the model stride, and applies peak normalization.


In [ ]:
def load_audio(path, target_sr=TARGET_SR):
    wave, sr = torchaudio.load(str(path))
    if wave.shape[0] > 1:
        wave = wave.mean(dim=0, keepdim=True)
    if sr != target_sr:
        wave = torchaudio.transforms.Resample(sr, target_sr)(wave)
    return wave, target_sr


def prepare_audio(wave, sr, clip_seconds=CLIP_SECONDS, start_seconds=START_SECONDS, factor=200):
    target_len = int(round(clip_seconds * sr)) if clip_seconds else wave.shape[-1]

    start = int(round(start_seconds * sr))
    stop = start + target_len
    clip = wave[..., start:stop]
    if clip.shape[-1] < target_len:
        clip = F.pad(clip, (0, target_len - clip.shape[-1]))

    usable_len = clip.shape[-1] - (clip.shape[-1] % factor)
    clip = clip[..., :usable_len]

    batch = Normalize1D()(clip.unsqueeze(0).clone())
    return batch


audio_path = resolve_input(AUDIO_SOURCE)
raw_wave, sample_rate = load_audio(audio_path)
orig = prepare_audio(raw_wave, sample_rate).to(device)

duration = orig.shape[-1] / sample_rate
peak = orig.abs().max().item()

print(
    f"Loaded audio: {display_path(audio_path)}  \nPrepared shape: {tuple(orig.shape)}  \nDuration: {duration:.2f}s  \nPeak after normalization: {peak:.3f}"
)



## 4. Run Codec Inference

A forward pass returns reconstructed waveform samples and RVQ code indices. The separate encode/decode path below exposes the same codec interface directly.


In [ ]:
with torch.no_grad():
    out = model(orig=orig)

recon = out["recon"].detach().cpu().clamp(-1, 1)
orig_cpu = orig.detach().cpu().clamp(-1, 1)
codes = out["codes"].detach().cpu()

bits_per_code = int(torch.ceil(torch.log2(torch.tensor(cfg.model.cb_size))).item())
kbps = codes.numel() * bits_per_code / 1000 / duration

print(
    f"Codes: {tuple(codes.shape)}  \nEstimated bitrate: {kbps:.2f} kbps  \nMean codebook perplexity: {out['mean_perplexity'].item():.2f}")


## 5. Listen

In [ ]:
print("Original")
display(Audio(orig_cpu[0, 0], rate=sample_rate))

print("Reconstruction")
display(Audio(recon[0, 0], rate=sample_rate))


## 6. Quick Visual Check

In [ ]:
seconds = torch.arange(orig_cpu.shape[-1]) / sample_rate
preview = min(orig_cpu.shape[-1], sample_rate * 2)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(seconds[:preview], orig_cpu[0, 0, :preview], linewidth=0.8)
axes[0].set_title("Original waveform")
axes[0].set_ylabel("amplitude")
axes[1].plot(seconds[:preview], recon[0, 0, :preview], linewidth=0.8, color="tab:orange")
axes[1].set_title("Reconstruction waveform")
axes[1].set_xlabel("seconds")
axes[1].set_ylabel("amplitude")
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 7. Save Demo Output

In [ ]:
output_flac = DEMO_DATA_DIR / "reconstruction.flac"
torchaudio.save(str(output_flac), recon[0], sample_rate)

print(f"Saved reconstruction to {display_path(output_flac)}")


## 8. Encode and Decode Separately

`model.gen` exposes the generator-only codec. Its `encode` method produces RVQ codes, and `decode` reconstructs audio from those codes.


In [ ]:
gen = model.gen
with torch.no_grad():
    out = gen.encode(orig=orig)

codes = out["codes"].detach().cpu()

print(f"Codes: {tuple(codes.shape)}\n")

The code tensor is the compact representation that can be stored or transmitted before decoding.


In [ ]:
out = gen.decode(codes)
recon_sep = out["recon"].detach().cpu().clamp(-1, 1)
print("Reconstruction")
display(Audio(recon_sep[0, 0], rate=sample_rate))

Compare the direct reconstruction with the reconstruction produced by separate encode/decode calls.


In [ ]:
eq = torch.equal(recon, recon_sep)
status = "equal" if eq else "different"
print(f"Direct and separate reconstructions are {status}.")
